<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c04-header.png" alt="Nextia Learning · SQL and Data Preparation for AI" width="100%">

# Solution: Handle messy values

**[Handle messy values](https://learning.nextia-ai.com/courses/sql/m04/handle-messy-values/)** · SQL and Data Preparation for AI · Module 4, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** decide for each problem in the data whether to exclude, correct or impute, write the decisions as one script, `scripts/clean.py`, keep the original values and a reason for every rejected row, and show that every input row is accounted for.

| | |
|---|---|
| **Time** | About 55 minutes |
| **Needs** | An internet connection for the setup cells. No account and no local install: the first cell installs pandas 3.0.6 in this notebook. |
| **You should know** | The problems that the profile found, from [Profile before changing](https://learning.nextia-ai.com/courses/sql/m04/profile-before-changing/). Loading with types, `merge` with `validate`, and changing rows with `.loc`, from [Transform with Python](https://learning.nextia-ai.com/courses/sql/m04/transform-with-python/). |
| **You do not need** | The local project, or your `clean.py` from lesson 2. The setup below builds the database and the extract, and this notebook writes the full `clean.py`. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/sql/m04/handle-messy-values/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M04-L03-handle-messy-values/handle-messy-values-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs `pandas` 3.0.6 in the versions that the course uses. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6
import pandas; print("pandas", pandas.__version__)

## Setup: get the practice data

Run the next cell. It downloads the three files of the Larkfield help-desk export (about 200 KB), checks each file's checksum, and builds the practice database, `larkfield.db`. It works in a folder `ticket-data`, laid out like your project from Module 1, so the code of the lesson runs here unchanged. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: customers 240, tickets 1417, outcomes 1874`. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-data` (in Colab: the **Files** panel on the left), then run the cell again.

This is the same database that `get_data.py` builds in [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database). The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the Larkfield export, check it, and build larkfield.db in
# ticket-data/. Standard library only: the same steps as get_data.py.
import csv, hashlib, json, os, sqlite3, subprocess, sys, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/"
FILES = {
    "customers.csv": "7b437642f0095524f8311f67a7caeb17100d23eed92618b9cf012cd3d3b79993",
    "tickets.csv": "9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28",
    "outcomes.csv": "67b4d9d9275b32061a8888050c351df7314054c11ee1ce29e856c1d5403d0653",
}
# The notebook works in its own folder ticket-data/, marked with a file, so it
# never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-data").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, expected):
    path = PROJECT / "data" / "raw" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-data and run again.")
    return path

DB = PROJECT / "larkfield.db"
if not DB.exists():
    paths = {name: fetch(name, sha) for name, sha in FILES.items()}
    build = sqlite3.connect(DB)
    build.executescript("""
        CREATE TABLE customers (customer_id TEXT, segment TEXT, region TEXT, joined_on TEXT);
        CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, created_at TEXT, channel TEXT, team TEXT,
            priority INTEGER, order_value REAL, word_count INTEGER, first_reply_minutes INTEGER,
            priority_now INTEGER, closed_at TEXT);
        CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, recorded_at TEXT, status TEXT,
            resolution_code TEXT, csat INTEGER);
        CREATE TABLE snapshot (name TEXT, value TEXT);
        INSERT INTO snapshot VALUES ('source', 'Larkfield help-desk export (synthetic, Nextia Learning C04)'),
            ('exported_at', '2026-07-01 06:00:00'), ('data_version', '1.0');
    """)
    for name, path in paths.items():
        with open(path, newline="", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        marks = ", ".join("?" for _ in rows[0])
        build.executemany(f"INSERT INTO {name[:-4]} ({', '.join(rows[0])}) VALUES ({marks})",
                          [[None if v == "" else v for v in row] for row in rows[1:]])
    build.commit()
    build.close()

os.chdir(PROJECT)  # work in the project folder, as in the lesson
for folder in ("queries", "scripts", "reports"):
    Path(folder).mkdir(exist_ok=True)
# Read-only, as q.py: a query cannot change the data by mistake.
con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
counts = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("customers", "tickets", "outcomes")]
print("Ready: customers {}, tickets {}, outcomes {}".format(*counts))

## Setup: rebuild the earlier lessons' files

This lesson starts from files that you made in earlier lessons: the extract `data/extract/` from [Reproducible extraction](https://learning.nextia-ai.com/courses/sql/m03/reproducible-extraction/). The next cell downloads the course's finished scripts ([C04/pipeline](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C04/pipeline)) into `ticket-data` and runs them in order, so you have the same files here. It takes about 20 seconds.

You should see one line per step, ending with `extract.py: done`. On your computer, your own scripts from those lessons make the same files.

In [ ]:
# Setup: rebuild the files of the earlier lessons with the course's scripts.
STEPS = ["extract"]
for file in ["queries/extract_tickets.sql", "queries/extract_outcomes.sql"] + [f"scripts/{s}.py" for s in STEPS]:
    if not Path(file).exists():
        download(LABS + "pipeline/" + file, Path(file))
for step in STEPS:
    result = subprocess.run([sys.executable, f"scripts/{step}.py"], capture_output=True, text=True)
    if result.returncode:
        raise SystemExit(f"{step}.py failed:\n{result.stdout}{result.stderr}")
    print(f"{step}.py: done")

## Setup: the helpers

The next cell defines small functions that the lesson uses. You do not need to read them, but you can.

- `sql(query)` runs a query and shows the result as a table, with the number of rows, like `q.py` and the boxes on the lesson page. It shows the first 20 rows; `sql(query, limit=None)` shows all of them.
- `check(query, ...)` compares the result of your query with the expected result, and says what is different. It does not show the answer.
- `expect(what, yours, expected)` compares one of your numbers with the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: show a query result, and check an answer without showing it.
try:
    import pandas as pd
except ImportError:
    pd = None

def run(query, db=None):
    cursor = (db or con).execute(query)
    columns = [d[0] for d in cursor.description] if cursor.description else []
    return columns, cursor.fetchall()

def sql(query, db=None, limit=20):
    """Run a query and show its result as a table, with the row count."""
    columns, rows = run(query, db)
    if not columns:
        print("No result: the cell has no SELECT yet. Write your query between the triple quotes, then run the cell again.")
        return
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        # Show NULL as NULL, as q.py and the boxes on the lesson page do.
        table = pd.DataFrame(shown, columns=columns, dtype=object)
        display(table.where(table.notna(), "NULL"))
    else:
        print("  ".join(columns))
        for row in shown:
            print("  ".join("NULL" if v is None else str(v) for v in row))
    more = f" (showing {len(shown)}; use limit=None to see all)" if len(shown) < len(rows) else ""
    print(f"{len(rows)} row{'s' if len(rows) != 1 else ''}{more}")

def fingerprint(rows, ordered=False):
    cells = [tuple("NULL" if v is None else f"{v:.6f}" if isinstance(v, float) else str(v) for v in r) for r in rows]
    return hashlib.sha256(repr(cells if ordered else sorted(cells)).encode()).hexdigest()[:16]

def check(query, rows, columns, expected, ordered=False, db=None):
    """Compare your result with the expected one: the column count, the row
    count, then the values (as a fingerprint, so the answer stays hidden)."""
    try:
        got_columns, got = run(query, db)
    except sqlite3.Error as error:
        print(f"Not yet: the query has an error: {error}")
        return
    if len(got_columns) != columns:
        print(f"Not yet: your result has {len(got_columns)} columns; the task asks for {columns}.")
    elif len(got) != rows:
        print(f"Not yet: your result has {len(got)} rows; the task expects {rows}.")
    elif fingerprint(got, ordered) != expected:
        print("Not yet: the number of rows is right, but some values" + (" or their order" if ordered else "") + " are different.")
    else:
        print("Check passed.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    same = abs(yours - expected) <= tolerance * max(1, abs(expected)) if isinstance(expected, float) else yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours}, which is not the expected value.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Three actions: exclude, correct, impute

For each problem in the profile, you have three possible actions:

- **Exclusion:** you take the row out of the clean table. You do not delete it: it goes to a **quarantine** table, with a **reject reason**, so a person can check it later.
- **Correction:** you change a value to the value that you **know** is correct, because a rule explains the error. The original stays in a second column.
- **Imputation:** you replace a missing value with an **estimate**, for example the median. The estimate is a guess, not a fact.

A fourth choice is often the best one: **keep the value missing**, and say why in a flag column.

How to choose:

- **Correct** only when you know the true value.
- **Keep missing** when missing has a meaning.
- **Quarantine** when the row is wrong and you cannot repair it.
- **Impute** only when the model needs a number, and the value exists but is unknown.

## 2. The decisions for Larkfield

Mei and Grace, the support lead, decide each problem. The counts are for the extract after the 14 exact duplicates are removed (1,402 rows):

| Problem | Rows | Action | New value | Original kept in |
|---|---|---|---|---|
| Exact duplicate rows | 14 | remove, and count | (row removed) | the count in `reports/cleaning.json` |
| Other spellings of `channel` | 39 | correct: `strip`, `lower`, `webform` → `web_form` | `email`, `chat`, `web_form` | `channel_raw` |
| `channel` is NULL | 38 | keep missing, as its own category | `unknown` | `channel_raw` (NULL) |
| `segment` is `Trade` | (customers) | correct: lower-case | `trade` | — |
| No `customer_id` (guest) | 12 | correct by meaning | segment `guest` | `customer_id` is still NULL |
| Customer not in `customers` | 9 | keep missing, as its own category | segment `unknown` | `customer_id` |
| `region` is NULL | 139 | keep missing, as its own category | `unknown` | — |
| `order_value` NULL, account team | 188 | keep missing | NULL, flag `not_applicable` | `order_value_raw` |
| `order_value` is `-1` | 11 | correct the sentinel to a real missing value | NULL, flag `unknown` | `order_value_raw` |
| Web form before 2026-02-01 | 24 | correct: divide by 100 | dollars, flag `converted_from_cents` | `order_value_raw` |
| Created before the customer joined | 3 | quarantine | — | `data/clean/quarantine.csv` |
| `word_count` above 10,000 | 1 | quarantine | — | `data/clean/quarantine.csv` |

The 11 `unknown` order values get an imputed median later, in Module 5, from the training rows only. The [lesson page](https://learning.nextia-ai.com/courses/sql/m04/handle-messy-values/#the-map-from-raw-to-clean) has the map from raw to clean, with the row count on each arrow.

## 3. Worked example: Mei writes clean.py

Mei writes the cleaning step in `scripts/clean.py`. This notebook writes the file in six parts, as the lesson shows it. The first cell makes the file: the loading code from lesson 2, with three new constants. Each constant holds a decision, with a name. Run the cell. It prints `Writing scripts/clean.py`.

In [ ]:
%%writefile scripts/clean.py
"""Clean the extracted tickets and outcomes. Every input row ends in exactly
one place: the clean table, the quarantine table, or the count of removed
exact duplicates. The original values stay next to the corrected ones."""

import json
from pathlib import Path

import pandas as pd

IN = Path("data/extract")
OUT = Path("data/clean")
REPORTS = Path("reports")

TICKET_TYPES = {
    "ticket_id": "string", "customer_id": "string", "channel": "string", "team": "string",
    "priority": "Int64", "order_value": "Float64", "word_count": "Int64",
    "first_reply_minutes": "Int64", "priority_now": "Int64",
    "segment": "string", "region": "string",
}
CHANNELS = {"email", "chat", "phone", "web_form"}
TIME_FORMAT = "%Y-%m-%d %H:%M:%S"
WEB_FORM_FIXED = pd.Timestamp("2026-02-01")  # the web form sent cents before this day
MAX_WORDS = 10_000


def load_tickets():
    df = pd.read_csv(IN / "tickets.csv", dtype=TICKET_TYPES, keep_default_na=False, na_values=[""])
    df["created_at"] = pd.to_datetime(df["created_at"], format=TIME_FORMAT)
    df["closed_at"] = pd.to_datetime(df["closed_at"], format=TIME_FORMAT)
    df["joined_on"] = pd.to_datetime(df["joined_on"], format="%Y-%m-%d")
    return df

Part 1 of `clean_tickets`: duplicates and categories. `%%writefile -a` adds the cell to the end of the file. Read the code, then run the cell. It prints `Appending to scripts/clean.py`.

- `assert df["ticket_id"].is_unique` checks the profile's finding: after removing exact copies, no ID is left twice.
- `assert set(df["channel"]) <= CHANNELS | {"unknown"}` checks that every channel is in the allowed list. A new spelling in a later export stops the script.
- The guests get `guest` first. Only the rows that are still missing after that get `unknown`: the 9 tickets of deleted customers.

In [ ]:
%%writefile -a scripts/clean.py


def clean_tickets(raw):
    counts = {"input_rows": len(raw)}
    df = raw.drop_duplicates()
    counts["exact_duplicates_removed"] = len(raw) - len(df)
    assert df["ticket_id"].is_unique, "a ticket ID appears twice with different values"

    # Channel: one spelling per value. A missing channel stays missing, and
    # becomes its own category, "unknown", so the model can still use the row.
    df["channel_raw"] = df["channel"]
    channel = df["channel"].str.strip().str.lower().replace({"webform": "web_form"})
    counts["channel_respelled"] = int((channel != df["channel"]).fillna(False).sum())
    df["channel"] = channel.fillna("unknown")
    assert set(df["channel"]) <= CHANNELS | {"unknown"}

    # Segment and region: the customer table spells "Trade" in two ways.
    # Guests have no customer; deleted customers have an ID but no row.
    df["segment"] = df["segment"].str.strip().str.lower()
    no_customer = df["customer_id"].isna()
    df.loc[no_customer, "segment"] = "guest"
    df["segment"] = df["segment"].fillna("unknown")
    df["region"] = df["region"].fillna("unknown")

Part 2: the order value. The order of the steps matters. The `cents` rule uses the **corrected** `channel`, so the ticket with `webform` (`T-10013`) is found too. The rule also skips missing values, so an account ticket from the web form stays missing. The last line counts each flag. Run the cell.

In [ ]:
%%writefile -a scripts/clean.py

    # Order value: keep the original, then correct or blank it, with a flag.
    df["order_value_raw"] = df["order_value"]
    df["order_value_flag"] = "ok"
    df.loc[df["team"] == "account", "order_value_flag"] = "not_applicable"
    unknown = df["order_value"] == -1
    df.loc[unknown, "order_value"] = pd.NA
    df.loc[unknown, "order_value_flag"] = "unknown"
    cents = (df["channel"] == "web_form") & (df["created_at"] < WEB_FORM_FIXED) & df["order_value"].notna()
    df.loc[cents, "order_value"] = (df.loc[cents, "order_value"] / 100).round(2)
    df.loc[cents, "order_value_flag"] = "converted_from_cents"
    counts.update({f"order_value_{k}": int(v) for k, v in df["order_value_flag"].value_counts().items()})

Part 3: quarantine and the row check. The last `assert` checks that clean + quarantined + removed duplicates equals the input. If a later change drops rows without a reason, the script stops. Run the cell.

In [ ]:
%%writefile -a scripts/clean.py

    # Rows we cannot correct with confidence go to quarantine, with a reason.
    df["reject_reason"] = pd.NA
    df.loc[df["created_at"] < df["joined_on"], "reject_reason"] = "created_before_customer_joined"
    df.loc[df["word_count"] > MAX_WORDS, "reject_reason"] = "word_count_above_10000"
    quarantine = df[df["reject_reason"].notna()]
    clean = df[df["reject_reason"].isna()].drop(columns="reject_reason")
    counts["quarantined"] = len(quarantine)
    counts["clean_rows"] = len(clean)
    assert counts["clean_rows"] + counts["quarantined"] + counts["exact_duplicates_removed"] == counts["input_rows"]
    return clean, quarantine, counts

The outcome events need two steps. The 30 repeated events (same ticket, time and status, but a new `outcome_id`) go: Mei keeps the one with the lowest `outcome_id`, the first one logged. The events of the quarantined tickets go too, because their ticket is not in the clean table. Run the cell.

In [ ]:
%%writefile -a scripts/clean.py


def clean_outcomes(tickets):
    df = pd.read_csv(IN / "outcomes.csv", dtype={"csat": "Int64"}, keep_default_na=False, na_values=[""])
    df["recorded_at"] = pd.to_datetime(df["recorded_at"], format=TIME_FORMAT)
    counts = {"input_rows": len(df)}
    # A retry logged some events twice: same ticket, time and status, new ID.
    # Keep the first one logged.
    df = df.sort_values("outcome_id").drop_duplicates(["ticket_id", "recorded_at", "status"], keep="first")
    counts["repeated_events_removed"] = counts["input_rows"] - len(df)
    df = df[df["ticket_id"].isin(tickets["ticket_id"])]
    counts["events_of_quarantined_tickets"] = counts["input_rows"] - counts["repeated_events_removed"] - len(df)
    counts["clean_rows"] = len(df)
    return df, counts

Last, `main()` writes three CSV files and the counts. `joined_on` is a day without a time: the two `strftime` lines write it as `2026-04-05`, not as `2026-04-05 00:00:00`. Run the cell.

In [ ]:
%%writefile -a scripts/clean.py


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    REPORTS.mkdir(exist_ok=True)
    tickets, quarantine, ticket_counts = clean_tickets(load_tickets())
    outcomes, outcome_counts = clean_outcomes(tickets)
    for df in (tickets, quarantine):
        df["joined_on"] = df["joined_on"].dt.strftime("%Y-%m-%d")
    tickets.to_csv(OUT / "tickets.csv", index=False, date_format=TIME_FORMAT)
    quarantine.to_csv(OUT / "quarantine.csv", index=False, date_format=TIME_FORMAT)
    outcomes.to_csv(OUT / "outcomes.csv", index=False, date_format=TIME_FORMAT)
    summary = {"tickets": ticket_counts, "outcomes": outcome_counts}
    (REPORTS / "cleaning.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()

> **Predict.** The extract has 1,416 ticket rows. How many clean ticket rows does the script write? Write it down, then run the script.

In [ ]:
!"{sys.executable}" scripts/clean.py

> **Check.** You should see `"clean_rows": 1398` for tickets and `"clean_rows": 1834` for outcomes. If an `assert` stops the script, the message names the check that failed: compare the parts with the lesson page.

Mei reconciles each group of numbers:

| Check | Calculation | Result |
|---|---|---|
| Ticket rows | 1,398 clean + 4 quarantined + 14 duplicates | 1,416 = the extract |
| Order value flags | 1,179 + 188 + 24 + 11 | 1,402: counted after the duplicates are removed and **before** the quarantine |
| Outcome events | 1,834 clean + 30 repeated + 5 of quarantined tickets | 1,869 = the extract |

The originals are still there. Mei looks at the quarantine file and at the first clean row with each flag. Run the cell to save the script.

In [ ]:
%%writefile scripts/show_clean.py
import pandas as pd

quarantine = pd.read_csv("data/clean/quarantine.csv")
columns = ["ticket_id", "created_at", "joined_on", "word_count", "reject_reason"]
print(quarantine[columns].to_string(index=False))

clean = pd.read_csv("data/clean/tickets.csv", keep_default_na=False, na_values=[""])
columns = ["ticket_id", "team", "order_value_raw", "order_value", "order_value_flag"]
first_of_each = clean.drop_duplicates("order_value_flag")
print(first_of_each[columns].to_string(index=False))

> **Predict.** How many rows are in the quarantine, and what is the reason for `T-10361`? Then run the script.

In [ ]:
!"{sys.executable}" scripts/show_clean.py

> **Check.** You should see 4 quarantined rows, each with a reason: `T-10361` (`word_count_above_10000`) and `T-11185`, `T-11224`, `T-11375` (`created_before_customer_joined`). Then one row per flag: `T-10001` `ok`, `T-10013` 3260.00 → 32.60 `converted_from_cents`, `T-10023` `not_applicable`, and `T-10046` -1.00 → `NaN` `unknown`.

`drop_duplicates("order_value_flag")` keeps the first row of each flag value. Anyone can now see what changed, check one row by hand, and undo a decision.

## 4. Failure case: filling missing values without knowing what they mean

`order_value` has 199 missing values in the clean table. A common reaction is "fill them with the mean". Mei shows Omar, the product manager, what that does. Run the cell to save the script.

In [ ]:
%%writefile scripts/try_fill.py
import pandas as pd

clean = pd.read_csv("data/clean/tickets.csv", keep_default_na=False, na_values=[""])
value = clean["order_value"]
print(f"rows: {len(clean)}, with a value: {value.notna().sum()}, missing: {value.isna().sum()}")
print(clean.loc[value.isna(), "order_value_flag"].value_counts())

mean = round(value.mean(), 2)
filled = value.fillna(mean)
account = clean["team"] == "account"
print(f"mean of the real values: {mean}")
print(f"account tickets with an order after filling: {filled[account].notna().sum()}")
print(f"orders invented for account tickets: {filled[account].sum():.2f} dollars")

> **Predict.** How many account tickets have an order after the fill? Then run the script.

In [ ]:
!"{sys.executable}" scripts/try_fill.py

> **Check.** You should see `missing: 199`, then `not_applicable 188` and `unknown 11`, a mean of `123.08`, `188` account tickets with an order, and `23139.04 dollars` of invented orders.

188 password resets now have an order of 123.08 dollars each. The column says "every ticket has an order", which is false. Only the 11 `unknown` values are real orders with an unknown amount.

**Fix:** group the missing values by a flag or a related column before you fill. Keep "not applicable" values missing, with a flag. Impute only values that exist but are unknown, with an indicator column, and learn the estimate from the training rows only (Module 5).

## 5. Your turn: one step in a different order

In part 2, the `cents` rule uses the corrected `channel`. What happens if it uses `channel_raw`, the original value?

> **Your turn.** In the next cell, change `changed_line` so that the rule uses `df["channel_raw"]`. Predict first: how many rows does the rule convert now, and which ticket is different? Run the cell, then run the two cells after it.

In [ ]:
# The line in part 2. Change the copy in changed_line: use channel_raw.
original_line = 'cents = (df["channel"] == "web_form") & (df["created_at"] < WEB_FORM_FIXED) & df["order_value"].notna()'
changed_line = 'cents = (df["channel_raw"] == "web_form") & (df["created_at"] < WEB_FORM_FIXED) & df["order_value"].notna()'

The next cell runs a copy of `clean.py` with your line, `scripts/clean_variant.py`, and compares its counts with the first run. Your `clean.py` does not change, so you do not need to change the line back. Run the cell.

In [ ]:
# Run a copy of clean.py with your line. It writes to data/variant/, so your
# clean.py and data/clean/ do not change.
import json
source =Path("scripts/clean.py").read_text(encoding="utf-8")
assert original_line in source, "clean.py does not have the original line. Run the cells of section 3 again."
variant = (source.replace(original_line, changed_line)
           .replace('OUT = Path("data/clean")', 'OUT = Path("data/variant")')
           .replace('REPORTS = Path("reports")', 'REPORTS = Path("data/variant")'))
Path("scripts/clean_variant.py").write_text(variant, encoding="utf-8")
result = subprocess.run([sys.executable, "scripts/clean_variant.py"], capture_output=True, text=True)
if result.returncode:
    print(result.stderr.strip().splitlines()[-1])
else:
    first = json.loads(Path("reports/cleaning.json").read_text())["tickets"]
    second = json.loads(Path("data/variant/cleaning.json").read_text())["tickets"]
    for name in first:
        mark = "" if first[name] == second.get(name) else "   <- different"
        print(f"{name:35} {first[name]:>6} {second.get(name, '-'):>6}{mark}")

In [ ]:
# Check: the order of the steps changes one count.
import json
variant_counts = json.loads(Path("data/variant/cleaning.json").read_text())["tickets"]
expect("order_value_converted_from_cents", variant_counts["order_value_converted_from_cents"], 23)
expect("clean_rows", variant_counts["clean_rows"], 1398)

When the check passes, `order_value_converted_from_cents` is 23, not 24, and `order_value_ok` is 1180, not 1179. `T-10013` has `channel_raw` = `webform`, so the rule does not see it, and its 3,260 stays as dollars. The row check still passes: it counts rows, not values. This is why the order of the steps is a decision to write down: "correct the channel first, then the units".

## 6. Module practice

### Worked example: seven messy tickets

Mei shows the method on seven made-up rows, so every intermediate result is visible. Each row has one problem.

> **Predict.** How many rows go to quarantine, and how many stay clean? Then run the cell.

In [ ]:
MESSY = """ticket_id,team,channel,created_at,joined_on,order_value,word_count
T-1,delivery,Email,2026-01-10 09:00:00,2025-06-01,42.5,80
T-2,account,chat,2026-03-02 11:30:00,2024-02-11,,45
T-3,returns,web_form,2026-03-03 14:00:00,2025-01-20,-1,120
T-3,returns,web_form,2026-03-03 14:00:00,2025-01-20,-1,120
T-4,payment,,2026-03-04 08:15:00,2026-03-10,64.5,60
T-5,warranty,phone,2026-03-05 16:45:00,2023-09-30,210,31000
T-6,delivery,web_form,2026-01-20 10:00:00,2025-11-05,4250,95
"""
import io

raw = pd.read_csv(io.StringIO(MESSY), dtype={"channel": "string", "order_value": "Float64"},
                  keep_default_na=False, na_values=[""])
raw["created_at"] = pd.to_datetime(raw["created_at"], format="%Y-%m-%d %H:%M:%S")
raw["joined_on"] = pd.to_datetime(raw["joined_on"], format="%Y-%m-%d")
print(f"1. input: {len(raw)} rows")

df = raw.drop_duplicates()
print(f"2. exact duplicates removed: {len(raw) - len(df)} -> {len(df)} rows")

df["channel_raw"] = df["channel"]
df["channel"] = df["channel"].str.strip().str.lower().fillna("unknown")

df["order_value_raw"] = df["order_value"]
df["order_value_flag"] = "ok"
df.loc[df["team"] == "account", "order_value_flag"] = "not_applicable"
unknown = df["order_value"] == -1
df.loc[unknown, "order_value"] = pd.NA
df.loc[unknown, "order_value_flag"] = "unknown"
cents = (df["channel"] == "web_form") & (df["created_at"] < "2026-02-01") & df["order_value"].notna()
df.loc[cents, "order_value"] = (df.loc[cents, "order_value"] / 100).round(2)
df.loc[cents, "order_value_flag"] = "converted_from_cents"

df["reject_reason"] = pd.NA
df.loc[df["created_at"] < df["joined_on"], "reject_reason"] = "created_before_customer_joined"
df.loc[df["word_count"] > 10_000, "reject_reason"] = "word_count_above_10000"
print(df[["ticket_id", "channel_raw", "channel", "order_value_raw", "order_value", "order_value_flag", "reject_reason"]].to_string(index=False))

quarantine = df[df["reject_reason"].notna()]
clean = df[df["reject_reason"].isna()]
print(f"3. quarantined: {len(quarantine)}, clean: {len(clean)}")
print(f"4. check: {len(clean)} + {len(quarantine)} + {len(raw) - len(df)} = {len(raw)}")

> **Check.** You should see `1. input: 7 rows`, `2. exact duplicates removed: 1 -> 6 rows`, a table of 6 tickets, `3. quarantined: 2, clean: 4` and `4. check: 4 + 2 + 1 = 7`.

Read the table row by row. T-1: the channel is corrected, the original stays. T-2: account, so the missing value stays missing. T-3: the copy is gone, and `-1` became a real missing value with the flag `unknown`. T-4 and T-5 go to quarantine, each with its reason. T-6: a January web form value, divided by 100.

pandas 3 shows no warning when the cell adds columns to `df`: `drop_duplicates()` gives a new table (copy-on-write), and the cell changes it with `df[...] =` and `.loc` only.

### Your task: the quality summary

Priya will train the model. She needs a short summary of the cleaning.

> **Your turn.** Complete `reports/quality.md` in the next cell, then run it. Answer these questions in short sections:
>
> 1. How many rows came in, and where did each one go?
> 2. Which values were corrected, how many, and where are the originals?
> 3. Which values are still missing, and what does each kind of missing value mean?
> 4. Which rows are in quarantine, and why?
> 5. What did you assume? Write each assumption that a correction depends on, and who confirmed it.
> 6. What is decided but not done yet?
>
> Take the counts from `reports/cleaning.json` (the output of `clean.py` above). Then run the check cell. It looks for the counts and IDs, not for your exact words.

In [ ]:
%%writefile reports/quality.md
# Quality summary: Larkfield tickets after cleaning

Input: data/extract/tickets.csv (1,416 rows) and data/extract/outcomes.csv
(1,869 rows), snapshot 2026-07-01 06:00. Script: scripts/clean.py.
All counts: reports/cleaning.json.

## Where every row went
| Tickets                              | Rows  |
|--------------------------------------|-------|
| data/clean/tickets.csv               | 1,398 |
| data/clean/quarantine.csv            | 4     |
| exact duplicates removed (counted)   | 14    |
| total = input                        | 1,416 |

Outcome events: 1,834 clean + 30 repeated events removed (kept the lowest
outcome_id) + 5 events of quarantined tickets = 1,869.

## Corrections (originals kept)
- channel: 39 respelled (strip, lower, webform -> web_form). Original: channel_raw.
- order_value: 24 web form values before 2026-02-01 divided by 100
  (flag converted_from_cents). Original: order_value_raw.
- order_value: 11 values of -1 set to missing (flag unknown). Original: order_value_raw.
- segment: lower-cased (Trade -> trade).

## Missing values and their meaning
- order_value missing, flag not_applicable (188): account tickets, no order. Do not fill.
- order_value missing, flag unknown (11): an order exists, amount unknown.
- channel "unknown" (38): not recorded.
- segment "guest" (12): no account. segment "unknown" (9): deleted customer.
- region "unknown": the customer gave no region, or there is no customer row.

## Quarantine (4 rows, data/clean/quarantine.csv)
- created_before_customer_joined: T-11185, T-11224, T-11375. One date is
  wrong; we cannot tell which.
- word_count_above_10000: T-10361 (48,210 words, a pasted log file).

## Assumptions behind the corrections
- The old web form sent cents until 2026-02-01, and only the web form (Grace).
- -1 in order_value means "not found in the order system" (Grace).
- Email, EMAIL and " chat" are the same channels as email and chat.
- Exact duplicate rows are copies from an export page overlap, not two tickets.
- 0 words means an attachment only, and is valid (Grace).

## Decided, not done yet
- The 11 unknown order values get the median with an indicator column,
  learned from the training set only (Module 5).
- Grace checks the 4 quarantined tickets by hand.

In [ ]:
# Check: does the summary give the counts? (It does not check your words.)
import re
text = Path("reports/quality.md").read_text(encoding="utf-8").replace(",", "")
needed = {
    "the input tickets": "1416", "the clean tickets": "1398", "the exact duplicates": "14",
    "the respelled channels": "39", "the values in cents": "24", "the -1 values": "11",
    "the account tickets with no order": "188", "the input events": "1869",
    "the clean events": "1834", "the repeated events": "30",
    "the ticket with a pasted log file": "T-10361", "the tickets created before the customer joined": "T-11185",
}
absent = [what for what, value in needed.items() if not re.search(rf"(?<![\w-]){value}(?![\w-])", text)]
print("Check passed." if not absent else "Not yet: the summary has no count or ID for " + "; ".join(absent) + ".")

The [lesson page](https://learning.nextia-ai.com/courses/sql/m04/handle-messy-values/#your-task-the-quality-summary) has a model answer. Compare your assumptions with it.

## Check your understanding and recap

Answer the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/sql/m04/handle-messy-values/#module-check). It counts toward your certificate when you are signed in and enrolled.

In this module you went from a raw export to a clean table that you can explain: profile first, load with types, then decide per problem. Quarantine with a reason, correct only with a known rule and keep the original, keep meaningful missing values missing with a flag, and impute later from training data only. Every row is counted: 1,416 = 1,398 + 4 + 14 tickets, and 1,869 = 1,834 + 30 + 5 events.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Features and target timing](https://learning.nextia-ai.com/courses/sql/m05/features-and-target-timing/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m04/handle-messy-values/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.